In [13]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
import plotly.express as px
import plotly.graph_objects as go
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from scipy.stats import pearsonr
import statsmodels.api as sm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import dotenv
from openai import OpenAI

In [5]:
df = pd.read_parquet("../../datasets/cleaned_datasets/demographic_infused_philly_c.parquet")
df['date'] = pd.to_datetime(df['date'])
print(df.head())
print("Columns:", df.columns)

                review_id                 user_id             business_id  \
0  J1LZjzbs5bFubvS135SD2g  5TE19zTjTIPq1HANACN7sw  dChRGpit9fM_kZK5pafNyA   
1  ecMiAOFucDM3zwXYfY-Q6A  5Z8S9OsHWCnE8wbxk1poQQ  s3Q1J4XEVOBiZy9dYUpqpg   
2  yuFQRhHo3z4TgE6drPXSgg  hcw7ndQKWGEH4P7BYAlG9w  JUlsvVAvZvGHWFfkKm0nlg   
3  Zdh0_HtE724MnohLOrB5Iw  OYaEBYLBrLY4mla8bOMbnA  9b0Mrvs6uJu2jJqet_Jwew   
4  y_XYEZk2Cin-q4N0czeaYw  _9VhEn9zaB-6txE3STNfLw  PYUI1OJVksGUbCrteU68bw   

   stars_rev                                               text  \
0          5  Had a great big meal with family and we loved ...   
1          5  Many locations.  All have lines so be prepared...   
2          5  Compliments to the chef and to the rest of the...   
3          4  I decided to try this spot out -- and it didn'...   
4          3  First off, finding parking is atrocious. Your ...   

                 date               name              address          city  \
0 2020-01-20 00:36:44           The Love        130 S 1

In [10]:
num_cols = len(df.columns)
start_index = num_cols - 37
end_index = num_cols - 1
columns_to_drop = ['text', 'postal_code']


# Drop the last 5 columns but keep the last column
df_trimmed = df.iloc[:, :start_index].join(df.iloc[:, end_index:])
df_trimmed = df_trimmed.drop(columns_to_drop, axis=1)
df_trimmed.columns
df = df_trimmed

# Sentiment Analysis

In [12]:
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['clean_text'].apply(lambda t: sia.polarity_scores(t)['compound'])
df['subjectivity_score'] = df['clean_text'].apply(lambda t: TextBlob(t).sentiment.subjectivity)

# Topic Modeling

In [14]:
reviews = df['clean_text'].dropna().tolist()
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=10)
tfidf = vectorizer.fit_transform(reviews)

n_topics = 10
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(tfidf)

def get_top_words(model, feature_names, n_top=10):
    topics = {}
    for idx, comp in enumerate(model.components_):
        top_indices = comp.argsort()[-n_top:][::-1]
        topics[idx] = [feature_names[i] for i in top_indices]
    return topics

feature_names = vectorizer.get_feature_names_out()
topics_words = get_top_words(lda, feature_names)

dotenv.load_dotenv(override=True)
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Please set OPENAI_API_KEY in your environment.")

client = OpenAI(api_key=api_key)

def label_topic(keywords):
    prompt = (
        "Below are keywords representing a topic from Philly Yelp reviews:\n"
        f"{', '.join(keywords)}\n\n"
        "Provide a concise label (evaluable) for this topic, e.g. 'Affordable Dining'."
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"system","content":"You are a helpful assistant."},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

topic_labels = {}
for tidx, words in topics_words.items():
    topic_labels[tidx] = label_topic(words)
print("Generated topic labels:", topic_labels)

# Assign each review to its top topic
doc_topic = lda.transform(tfidf)
df = df.loc[df['clean_text'].notna()].copy()
df['predicted_topic'] = np.argmax(doc_topic, axis=1)
df['topic_label'] = df['predicted_topic'].map(topic_labels)


Generated topic labels: {0: '"Delicious Diverse Cuisine"', 1: "'Wedding Services'", 2: "'Philly Food Favorites'", 3: "'Delivery Service Experience'", 4: "'Casual Dining Experience'", 5: "'Quality Food and Coffee'", 6: "'Sushi Restaurant Experience'", 7: "'Auto Repair Services'", 8: "'Excellent Dining Experience'", 9: 'Beauty and Wellness Services'}


# Geospatial Clustering of Businesses


In [15]:
if not {'latitude','longitude'}.issubset(df.columns):
    raise KeyError("latitude/longitude missing.")

coords = df[['latitude','longitude']].to_numpy()
coords_rad = np.radians(coords)

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, metric='haversine')
df['hdbscan_cluster'] = clusterer.fit_predict(coords_rad)

c:\Users\alche\OneDrive\Documents\GitHub\AllenCheung0213.github.io\CS554_NLP_Final_Project\newenv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\alche\OneDrive\Documents\GitHub\AllenCheung0213.github.io\CS554_NLP_Final_Project\newenv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [16]:
gdf_biz = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)
gdf_biz.to_file("gdf_businesses.geojson", driver="GeoJSON")
print("Saved: gdf_businesses.geojson")

Saved: gdf_businesses.geojson


In [17]:
valid = gdf_biz[gdf_biz.hdbscan_cluster != -1]
cluster_metrics = (
    valid
    .groupby('hdbscan_cluster')
    .agg(
        avg_sentiment    = ('sentiment_score','mean'),
        avg_subjectivity = ('subjectivity_score','mean'),
        avg_stars        = ('stars_rev','mean'),
        business_count   = ('hdbscan_cluster','count')
    )
    .reset_index()
)

clusters = valid.dissolve(by='hdbscan_cluster', as_index=False)
clusters['geometry'] = clusters.geometry.convex_hull
clusters = clusters.merge(cluster_metrics, on='hdbscan_cluster')

clusters.to_file("clusters.geojson", driver="GeoJSON")
print("Saved: clusters.geojson")

Saved: clusters.geojson


# Statistical Analysis

In [19]:
postal_agg = df.groupby('area_code').agg(
    avg_sentiment=('sentiment_score', 'mean'),
    avg_subjectivity=('subjectivity_score', 'mean'),
    avg_stars=('stars_rev', 'mean'),
    count=('area_code', 'count')
).reset_index()
print("Postal Code Aggregation:")
print(postal_agg)

rho, pval = pearsonr(postal_agg['avg_sentiment'], postal_agg['avg_stars'])
print(f"Spearman correlation (sentiment vs. stars) by postal code: {rho:.2f} (p={pval:.3f})")

X = postal_agg[['avg_stars', 'count']]
X = sm.add_constant(X)
y = postal_agg['avg_sentiment']
model = sm.OLS(y, X).fit()
print(model.summary())
with open("regression_summary.txt", "w") as f:
    f.write(model.summary().as_text())
print("Regression summary saved to regression_summary.txt")

Postal Code Aggregation:
   area_code  avg_sentiment  avg_subjectivity  avg_stars  count
0      19102       0.688308          0.570692   3.817379   4419
1      19103       0.709441          0.578621   3.968423  11749
2      19104       0.614216          0.571433   3.616462   4568
3      19106       0.734416          0.575720   4.024276   7044
4      19107       0.700751          0.574740   3.936888  16843
5      19108       0.062275          0.613058   2.500000      8
6      19109       0.383738          0.481700   3.750000      8
7      19111       0.427476          0.543402   3.291304    690
8      19112       0.451557          0.544856   3.327586     58
9      19113       0.550281          0.596625   3.592593     27
10     19114       0.469363          0.538518   3.244280   1355
11     19115       0.458083          0.546893   3.289116    882
12     19116       0.534977          0.562654   3.631649    752
13     19118       0.635535          0.569761   3.618243    888
14     19119   